In [1]:
import pandas as pd

df = pd.read_csv("../data/security_events.csv", parse_dates=["timestamp"])
df.head()

,event_id,timestamp,event_type,severity,source_ip,country,username,host,action_taken,status
0,1,2026-07-08 00:05:17,firewall_block,medium,10.1.5.54,Internal,emartin,SRV-08,blocked,resolved
1,2,2026-07-08 00:24:19,successful_login,low,45.33.32.248,USA,asmith,WS-013,allowed,resolved
2,3,2026-07-08 00:30:25,successful_login,medium,10.1.18.42,Internal,garcia,WS-001,allowed,resolved
3,4,2026-07-08 00:36:58,successful_login,low,10.1.18.24,Internal,bjones,SRV-01,allowed,resolved
4,5,2026-07-08 01:06:44,malware_alert,critical,45.33.32.190,USA,-,WS-025,quarantined,resolved


In [2]:
print(df.shape)      # (rows, columns)
print(df.dtypes)     # data type of each column
df.isna().sum()      # count of missing values per column

(10000, 10)
event_id                 int64
timestamp       datetime64[us]
event_type                 str
severity                   str
source_ip                  str
country                    str
username                   str
host                       str
action_taken               str
status                     str
dtype: object


event_id        0
timestamp       0
event_type      0
severity        0
source_ip       0
country         0
username        0
host            0
action_taken    0
status          0
dtype: int64

In [3]:
df["event_type"].value_counts()
df["severity"].value_counts(normalize=True)   # as proportions
df["action_taken"].value_counts()

action_taken
allowed        4498
logged         2524
blocked        1524
alerted         677
quarantined     491
locked_out      286
Name: count, dtype: int64

In [4]:
df["date"] = df["timestamp"].dt.floor("D")   # date without the time part
df["hour"] = df["timestamp"].dt.hour         # 0-23, for "when do attacks happen?"

df["source_ip"].value_counts().head(10)      # top 10 source IPs
df.loc[df["event_type"] == "failed_login", "username"].value_counts().head(10)

username
-                394
administrator    375
root             374
svc_backup       345
test             342
admin            341
clee              53
bjones            53
fnguyen           45
dpatel            44
Name: count, dtype: int64

In [5]:
import plotly.express as px

daily = df.groupby("date").size().reset_index(name="events")
fig = px.line(daily, x="date", y="events", title="Security events per day")
fig.show()

type_counts = df["event_type"].value_counts().reset_index(name="count")
px.bar(type_counts, x="event_type", y="count", title="Events by type").show()

sev_by_hour = df.groupby(["hour", "severity"]).size().reset_index(name="count")
px.line(sev_by_hour, x="hour", y="count", color="severity",
        title="Events by hour of day").show()

In [6]:
df.to_csv("../data/security_events_clean.csv", index=False)
print("saved", df.shape)

saved (10000, 12)
